<a href="https://colab.research.google.com/github/Ayush-Singh-36/fnn_Pytorch_Model/blob/main/fnn_pytorch_model_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

## Device-Agnostic code

In [45]:
import torch
from torch import cuda
if cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
print(device)

cuda


## Calling the dataset from kaggle, directly here

In [46]:
import os
from google.colab import userdata
import sys
def custom_exit(status):
    print(f"Kaggle API tried to exit with status {status}. Ignoring for Colab environment.")
sys.exit = custom_exit
exit = custom_exit
try:
    __builtins__.exit = custom_exit
except AttributeError:
    print("Could not patch __builtins__.exit - it might not be present or modifiable in this environment.")

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "nudratabbas/software-developer-salary-prediction-dataset"
download_path = "./data"

print("Authenticating via environment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading movie dataset from Kaggle...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")


Authenticating via environment variables...
Dataset URL: https://www.kaggle.com/datasets/nudratabbas/software-developer-salary-prediction-dataset
Done! Your files have been saved to the './data' folder.


## Checking for cardinality

### data_dictionary.csv

In [47]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

df = pd.read_csv("./data/data_dictionary.csv")
profile_dataset_features(df)

=== Dataset Shape: 7 rows | 3 columns ===

Found 3 Categorical columns and 0 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'Column' | Unique Values Count: 7 | Missing: 0 rows
  - [experience]: 1 occurrences (14.29%)
  - [country]: 1 occurrences (14.29%)
  - [education]: 1 occurrences (14.29%)
  - [languages]: 1 occurrences (14.29%)
  - [frameworks]: 1 occurrences (14.29%)
  - [company_size]: 1 occurrences (14.29%)
  - [salary_usd]: 1 occurrences (14.29%)

🔹 Feature: 'Type' | Unique Values Count: 3 | Missing: 0 rows
  - [string]: 5 occurrences (71.43%)
  - [number]: 1 occurrences (14.29%)
  - [target]: 1 occurrences (14.29%)

🔹 Feature: 'Description' | Unique Values Count: 7 | Missing: 0 rows
  - [Years of professional coding experience]: 1 occurrences (14.29%)
  - [Country of residence]: 1 occurrences (14.29%)
  - [Highest level of formal education]: 1 occurrenc

### train.csv

In [48]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

train_df = pd.read_csv("./data/train.csv")
profile_dataset_features(train_df)

=== Dataset Shape: 40000 rows | 7 columns ===

Found 5 Categorical columns and 2 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'country' | Unique Values Count: 10 | Missing: 0 rows
  - [USA]: 16003 occurrences (40.01%)
  - [UK]: 4013 occurrences (10.03%)
  - [Canada]: 4003 occurrences (10.01%)
  - [Germany]: 3988 occurrences (9.97%)
  - [India]: 3970 occurrences (9.93%)
  - [Australia]: 2064 occurrences (5.16%)
  - [France]: 2031 occurrences (5.08%)
  - [Japan]: 1918 occurrences (4.79%)
  - [Brazil]: 1030 occurrences (2.57%)
  - [Singapore]: 980 occurrences (2.45%)

🔹 Feature: 'education' | Unique Values Count: 5 | Missing: 0 rows
  - [Bachelors]: 20061 occurrences (50.15%)
  - [Masters]: 11968 occurrences (29.92%)
  - [Some College]: 4005 occurrences (10.01%)
  - [High School]: 1988 occurrences (4.97%)
  - [PhD]: 1978 occurrences (4.95%)

🔹 Feature: 'languages'

,min,max,mean
experience,0.0,40.0,19.912875
salary_usd,12024.0,277554.0,131834.441525


### test.csv

In [49]:
import pandas as pd
import numpy as np
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    """
    Scans a massive dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries.
    """
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
        display(numeric_summary)
    else:
        print("No numerical columns to describe.")

test_df = pd.read_csv("./data/test.csv")
profile_dataset_features(test_df)

=== Dataset Shape: 10000 rows | 7 columns ===

Found 5 Categorical columns and 2 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'country' | Unique Values Count: 10 | Missing: 0 rows
  - [USA]: 3999 occurrences (39.99%)
  - [Germany]: 1044 occurrences (10.44%)
  - [India]: 1037 occurrences (10.37%)
  - [UK]: 997 occurrences (9.97%)
  - [Canada]: 935 occurrences (9.35%)
  - [Japan]: 516 occurrences (5.16%)
  - [Australia]: 506 occurrences (5.06%)
  - [France]: 502 occurrences (5.02%)
  - [Brazil]: 251 occurrences (2.51%)
  - [Singapore]: 213 occurrences (2.13%)

🔹 Feature: 'education' | Unique Values Count: 5 | Missing: 0 rows
  - [Bachelors]: 5077 occurrences (50.77%)
  - [Masters]: 2995 occurrences (29.95%)
  - [Some College]: 990 occurrences (9.90%)
  - [High School]: 481 occurrences (4.81%)
  - [PhD]: 457 occurrences (4.57%)

🔹 Feature: 'languages' | Unique Val

,min,max,mean
experience,0.0,40.0,19.7701
salary_usd,12032.0,280900.0,131110.6842


## Creating dataloader out of dataframes

In [50]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler

categorical_feature_cols = ['country', 'education', 'languages', 'frameworks', 'company_size']
numerical_feature_cols = ['experience_scaled']

scaler = StandardScaler()
train_df["experience_scaled"] = scaler.fit_transform(train_df[["experience"]])
test_df["experience_scaled"] = scaler.transform(test_df[["experience"]])

cat_to_int_maps = {}
emb_dim_num = []
for col in categorical_feature_cols:
    all_unique_values = pd.concat([train_df[col], test_df[col]]).astype(str).unique()
    mapping = {category: i for i, category in enumerate(all_unique_values)}
    cat_to_int_maps[col] = mapping

    num_embeddings = len(all_unique_values)
    emb_dim = min(50, num_embeddings // 2)
    emb_dim_num.append((num_embeddings, emb_dim))

print("Categorical to Integer Maps:")
for col, mapping in cat_to_int_maps.items():
    print(f"  {col}: {len(mapping)} unique categories")
print("\nEmbedding Dimensions (num_embeddings, embedding_dim):")
print(emb_dim_num)


class MyDataset(Dataset):
  def __init__(self, dataframe, categorical_cols, numerical_cols, cat_to_int_maps, target_col="salary_usd"):
    self.labels = torch.tensor(dataframe[target_col].to_numpy(dtype=np.float32)).unsqueeze(1)

    x_cat_data = {}
    for col in categorical_cols:
        x_cat_data[col] = dataframe[col].map(cat_to_int_maps[col]).fillna(0).astype(int)

    self.x_cat = torch.tensor(pd.DataFrame(x_cat_data).to_numpy(dtype=np.int64))

    self.x_num = torch.tensor(dataframe[numerical_cols].to_numpy(dtype=np.float32))

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return (self.x_cat[idx], self.x_num[idx]), self.labels[idx]


train_dataset = MyDataset(train_df, categorical_feature_cols, numerical_feature_cols, cat_to_int_maps)
test_dataset = MyDataset(test_df, categorical_feature_cols, numerical_feature_cols, cat_to_int_maps)

train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

Categorical to Integer Maps:
  country: 10 unique categories
  education: 5 unique categories
  languages: 100 unique categories
  frameworks: 100 unique categories
  company_size: 6 unique categories

Embedding Dimensions (num_embeddings, embedding_dim):
[(10, 5), (5, 2), (100, 50), (100, 50), (6, 3)]


**Encoding**

In [51]:
import torch
import torch.nn as nn
embedding_layer = nn.ModuleList([
    nn.Embedding(num_embeddings = 7, embedding_dim = emb_dim)
    for num_embeddings, emb_dim in emb_dim_num
])

**Model Archtecture**

In [52]:
import torch
import torch.nn as nn
class SalaryPredictionModel(nn.Module):
  def __init__(self, emb_dim_num, numerical = 1):
    super().__init__()
    self.embeddings = nn.ModuleList([
        nn.Embedding(num_embeddings = num_embeddings, embedding_dim = emb_dim)
        for num_embeddings, emb_dim in emb_dim_num
    ])
    total_emb_dim = sum(emb_dim for _, emb_dim in emb_dim_num)
    in_features = total_emb_dim + numerical
    self.fc_net = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(64, 32),
        nn.ReLU(),

        nn.Linear(32, 1)
    )
  def forward(self, x_cat, x_num):
    embedded_outputs = [
        emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
    x_emb = torch.cat(embedded_outputs, dim = 1)
    x = torch.cat([x_emb, x_num], dim = 1)
    return self.fc_net(x)

**Loss Function & Optimizer**

In [64]:
criterion = nn.MSELoss()
model = SalaryPredictionModel(emb_dim_num)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

**Training Loop**

In [65]:
num_epochs = 100
for epoch in range(num_epochs):
  model.train()
  total_train_loss = 0
  for batch_idx, ((x_cat, x_num), targets) in enumerate(train_dataloader):
    x_cat, x_num, targets = x_cat.to(device), x_num.to(device), targets.to(device)
    optimizer.zero_grad()
    outputs = model(x_cat, x_num)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    total_train_loss += loss.item()

  avg_train_loss = total_train_loss / len(train_dataloader)
  train_rmse = torch.sqrt(torch.tensor(avg_train_loss))

  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Train RMSE: {train_rmse:.4f}")

Epoch 10/100, Train Loss: 415209312.5376, Train RMSE: 20376.6855
Epoch 20/100, Train Loss: 409367264.0000, Train RMSE: 20232.8262
Epoch 30/100, Train Loss: 400226409.2288, Train RMSE: 20005.6602
Epoch 40/100, Train Loss: 402100767.4560, Train RMSE: 20052.4512
Epoch 50/100, Train Loss: 394798676.5824, Train RMSE: 19869.5410
Epoch 60/100, Train Loss: 393656188.6656, Train RMSE: 19840.7715
Epoch 70/100, Train Loss: 383733706.0864, Train RMSE: 19589.1211
Epoch 80/100, Train Loss: 379762629.5040, Train RMSE: 19487.5000
Epoch 90/100, Train Loss: 375170113.1200, Train RMSE: 19369.3086
Epoch 100/100, Train Loss: 370041929.8304, Train RMSE: 19236.4746


**Saving Model Artifacts**

In [66]:
import torch.onnx

model.eval()

x_cat_example, x_num_example = train_dataset[0][0]

x_cat_example = x_cat_example.unsqueeze(0).to(device)
x_num_example = x_num_example.unsqueeze(0).to(device)

onnx_file_path = "salary_prediction_model.onnx"

try:
    torch.onnx.export(
        model,
        (x_cat_example, x_num_example),
        onnx_file_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['x_cat_input', 'x_num_input'],
        output_names=['output'],
        dynamic_axes={
            'x_cat_input': {0: 'batch_size'},
            'x_num_input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )
    print(f"Model successfully exported to {onnx_file_path}")
except Exception as e:
    print(f"Error exporting model to ONNX: {e}")


Error exporting model to ONNX: No module named 'onnxscript'
